# 🤖 Notebook 4: Manipulator Control & Architecture Comparison

In this notebook, we transition from single-axis motor jogging to controlling a coordinated **6-DOF multi-axis robotic arm (Dobot CR3A)** [cite: 124]. 
You will examine and execute motion control under WMX R2’s two primary architectural paradigms: **Proprietary Trajectory Control** and **Standard ROS 2 Control** [cite: 382].

---

## 🔄 Understanding the Two Control Architectures

To control a complex manipulator, WMX R2 provides two distinct pathways with unique engineering trade-offs [cite: 382]:

### 1. Proprietary Trajectory Control (`wmx_r2_package` Mode)
* **How it works**: `MoveIt 2` plans a path and sends a list of joint trajectory waypoints [cite: 373]. The custom `joint_trajectory_controller` in `wmx_r2_package` intercepts these points and hands them directly to the **WMX3 core engine**. WMX3 then executes its proprietary **C-Spline interpolation algorithms** to smoothly drive the EtherCAT motors.
* **Key Benefit**: Extremely lightweight on CPU resources, and leverages decades of proven industrial-grade motion profiles with microsecond synchronization.

### 2. Standard ROS 2 Control (`wmx_r2_control` Mode)
* **How it works**: The execution loop is completely governed by the ROS 2 standard `controller_manager`. WMX3 is loaded purely as a low-level **Hardware Interface Plugin** (`WmxSystemHardware`). The standard ROS 2 controller reads encoder feedback via `read()` and injects raw target commands via `write()` at a real-time periodic rate.
* **Key Benefit**: Provides a **"Zero-Gap Digital Twin"** workflow. The high-level 

In [2]:
import rclpy
from rclpy.node import Node
from movensys_manipulator_moveit_config.srv import MoveJoints  # Service for absolute joint movements

# 1. Initialize ROS 2 communication context (guaranteed run once)
if not rclpy.ok():
    rclpy.init()

# 2. Create a temporary client node to send motion commands
node = Node('notebook4_joints_client')
client = node.create_client(MoveJoints, '/api/move/joint_absolute')

print("📡 Connecting to trajectory planning service...")
if client.wait_for_service(timeout_sec=5.0):
    # 3. Formulate the absolute target pose (Safe Stance Pose)
    req = MoveJoints.Request()
    req.joint_names = ['joint1', 'joint2', 'joint3', 'joint4', 'joint5', 'joint6']
    req.joint_values = [0.0, 0.704, -1.989, -0.287, 1.57, 0.0]  # Safe target angles in radians
    
    print("🎯 [EXECUTE] Sending absolute stance pose command to Dobot CR3A...")
    
    # 4. Call the service and wait for the motion block to complete
    future = client.call_async(req)
    rclpy.spin_until_future_complete(node, future)
    
    res = future.result()
    if res is not None:
        print(f"✅ [SUCCESS] Motion Execution: {res.success} | Message: {res.message}")
    else:
        print("❌ [ERROR] Service call failed during execution.")
else:
    print("❌ [TIMEOUT] Service '/api/move/joint_absolute' not found.")
    print("👉 Please verify that Terminal 2 (trajectory_service.launch.py) is running successfully.")

# Clean up the temporary node
node.destroy_node()


📡 Connecting to trajectory planning service...
❌ [TIMEOUT] Service '/api/move/joint_absolute' not found.
👉 Please verify that Terminal 2 (trajectory_service.launch.py) is running successfully.
